# Taller 1: Econometría

- **Profesor:** Francisco Alfaro Medina
- **Ayudantes:** Krischnna Cortez y Karen Rojas


### Instrucciones

- Dispone de **60 minutos** para completar los **100 puntos** del taller.
- Cuide la presentación y redacción de sus respuestas.
- Puede utilizar su computador y los apuntes de clase y ayudantía.
- Debe entregar un archivo **PDF** y un **R script** (extensión `.R`).

> ⚠️ **Importante para Colab:** Este notebook usa un kernel de R. Si abre este archivo en Google Colab, seleccione **Runtime → Change runtime type → R** antes de ejecutar cualquier celda.

---
# Sección 1: Datos Aleatorios *(30 puntos)*

En esta sección trabajaremos con un dataset **simulado** de 50 alumnos de la USM. El dataset contiene las siguientes variables:

| Variable | Descripción |
|---|---|
| `hrs_sueno` | Horas de sueño promedio en el último mes |
| `profesor_part` | Si recibió ayuda de profesor particular (0/1) |
| `media_sem_pasado` | Promedio de notas del semestre anterior |
| `tiempo_est` | Horas de estudio dedicadas |
| `asistencia` | Porcentaje de asistencia a clases |
| `nivel_socioec` | Nivel socioeconómico (1 al 5) |
| `notas` | **Variable dependiente** — nota del alumno |

## Pregunta 1.1 — Generar el dataset *(6 pts.)*

Antes de ejecutar el código, **cambie la semilla** según la primera letra de su apellido:

| A–E | F–J | K–O | P–T | U–Z |
|:---:|:---:|:---:|:---:|:---:|
| 123 | 456 | 789 | 101112 | 131415 |

Reemplace el valor en `set.seed(...)` antes de continuar.

In [1]:
# -------------------------------------------------------
# Pregunta 1.1: Generar el dataframe "datos"
# Cambie la semilla según la primera letra de su apellido
# A-E: 123 | F-J: 456 | K-O: 789 | P-T: 101112 | U-Z: 131415
# -------------------------------------------------------

set.seed(101112)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO

datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

# Calcular notas con ponderaciones definidas
datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

# Asegurar rango entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,6.1,0,80.0,2.6,79.4,4,76.53174
2,5.3,0,83.8,7.7,61.8,4,95.09891
3,9.6,1,82.2,4.4,68.5,5,97.45453
4,5.7,1,72.7,3.5,98.6,1,88.80463
5,7.1,0,76.2,1.9,85.0,3,69.74552
6,8.3,1,75.7,7.4,85.3,5,100.00000


## Pregunta 1.2 — Redondear notas *(2 pts.)*

Redondee la variable `notas` a **1 decimal**.

In [9]:
# Pregunta 1.2: Redondear notas a 1 decimal
datos$notas <- round(datos$notas, 1)
head(datos$notas)
print(datos)

[1]  76.5  95.1  97.5  88.8  69.7 100.0

   hrs_sueno profesor_part media_sem_pasado tiempo_est asistencia nivel_socioec
1        6.1             0             80.0        2.6       79.4             4
2        5.3             0             83.8        7.7       61.8             4
3        9.6             1             82.2        4.4       68.5             5
4        5.7             1             72.7        3.5       98.6             1
5        7.1             0             76.2        1.9       85.0             3
6        8.3             1             75.7        7.4       85.3             5
7        6.6             0             93.7        7.0       79.9             1
8        9.4             1             70.8        6.3       93.0             4
9        9.5             0             84.1        1.2       83.5             5
10       8.2             0             67.1        2.6       78.5             1
11       5.2             1             92.8        7.3       60.8             2
12       7.9             0             9

## Pregunta 1.3 — Estimadores β via álgebra matricial *(8 pts.)*

Calcule los estimadores MCO **manualmente**, usando la fórmula matricial:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Sin usar** la función `lm()` de R.

In [29]:
# Pregunta 1.3: Estimadores beta via álgebra matricial (sin lm)
y <- as.matrix(datos$notas)

# variable (X)
X <- as.matrix(cbind(
  1,  # interceptocolumnas
  datos$hrs_sueno,
  datos$profesor_part,
  datos$media_sem_pasado,
  datos$tiempo_est,
  datos$asistencia,
  datos$nivel_socioec
))

# Cálculo beta
beta_hat <- solve(t(X) %% X) %% t(X) %*% y

# Mostrar resultados
print(beta_hat)

ERROR: Error in t(X)%%X: non-conformable arrays


## Pregunta 1.4 — Modelo con `lm()` e interpretación *(8 pts.)*

Genere el modelo de regresión múltiple usando la función `lm()` y obtenga el resumen con `summary()`.  
Luego, **interprete cada coeficiente** en el espacio indicado.

> 💡 **Tip:** Los β de `lm()` deben coincidir con los calculados manualmente en la pregunta anterior.

In [ ]:
# Pregunta 1.4: Modelo con lm() y summary
lm

## Pregunta 1.5 — Relación entre betas y ponderaciones del código *(6 pts.)*

Analice la relación entre los coeficientes estimados (β̂) y las ponderaciones reales usadas en el código del Anexo para generar las notas.



In [ ]:
# Pregunta 1.5: Comparación entre betas estimados y ponderaciones reales


# Sección 2: Wooldridge *(70 puntos)*

Para esta sección utilizaremos el paquete `wooldridge`, que contiene bases de datos clásicas de econometría.

In [13]:
# Instalar y cargar el paquete wooldridge (solo necesario la primera vez en Colab)
if (!require(wooldridge)) install.packages("wooldridge")
library(wooldridge)

---
## 2A. Base `wage1` *(30 puntos)*

El modelo de regresión poblacional a estimar es:

$$wage = \beta_0 + \beta_1\, educ + \beta_2\, exper + \beta_3\, tenure + u$$

Donde:
- `wage` = salario por hora (USD)
- `educ` = años de escolaridad
- `exper` = años de experiencia laboral
- `tenure` = años en el trabajo actual

In [16]:
# Cargar y limpiar la base wage1
data("wage1")
wage1 <- na.omit(wage1)

### Pregunta 2A.1 — Estimadores MCO e interpretación *(8 pts.)*

Estime el modelo completo e interprete los resultados.

In [19]:
# Pregunta 2A.1: Modelo de regresión múltiple con wage1
modelo <- lm(wage ~ educ + exper + tenure, data = wage1)
summary(modelo)



Call:
lm(formula = wage ~ educ + exper + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.6068 -1.7747 -0.6279  1.1969 14.6536 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.87273    0.72896  -3.941 9.22e-05 ***
educ         0.59897    0.05128  11.679  < 2e-16 ***
exper        0.02234    0.01206   1.853   0.0645 .  
tenure       0.16927    0.02164   7.820 2.93e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.084 on 522 degrees of freedom
Multiple R-squared:  0.3064,	Adjusted R-squared:  0.3024 
F-statistic: 76.87 on 3 and 522 DF,  p-value: < 2.2e-16


El intercepto 2.87273 representa el salario esperado cuando educ, exper y tenure valen cero.
El coeficiente de educ indica que, manteniendo constantes las demás variables, un año adicional de escolaridad aumenta el salario por hora en 0.599 USD.
El coeficiente de exper indica que, manteniendo constantes las demás variables, un año adicional de experiencia laboral aumenta el salario por hora en 0.022 USD.
El coeficiente de tenure indica que, manteniendo constantes las demás variables, un año adicional de antigüedad en el trabajo actual aumenta el salario por hora en 0.169 USD.

### Pregunta 2A.2 — ¿Los signos son los esperados? *(10 pts.)*

Antes de ver los resultados, reflexione: ¿qué signo debería tener cada coeficiente económicamente?

In [ ]:
# Pregunta 2A.2: Revisar signos de los coeficientes

Sí, los signos de los coeficientes son los esperados económicament, salvo el primer indicador que es negativo, siendo esto incongrüente con una situacion de salario, es decir, perder dinero por hora sin tener los demás variables presentes.


### Pregunta 2A.3 — Comparar R² y R² ajustado *(12 pts.)*

Estime un segundo modelo usando solo `educ` y `tenure`, y compare el ajuste con el modelo completo.

En el modelo reducido, el R2 es 0.3019, mientras que en el modelo completo era 0.3064. Esto muestra que el modelo completo explica una proporción levemente mayor de la variación del salario.

In [20]:
# Pregunta 2A.3: Modelo reducido (sin exper)
modelo_red <- lm(wage ~ educ + tenure, data = wage1)
summary(modelo_red)


Call:
lm(formula = wage ~ educ + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-8.1438 -1.7288 -0.6372  1.2575 14.7482 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.22162    0.64015   -3.47 0.000563 ***
educ         0.56914    0.04881   11.66  < 2e-16 ***
tenure       0.18958    0.01871   10.13  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.092 on 523 degrees of freedom
Multiple R-squared:  0.3019,	Adjusted R-squared:  0.2992 
F-statistic: 113.1 on 2 and 523 DF,  p-value: < 2.2e-16


---
## 2B. Base `attend` *(40 puntos)*

En esta sección analizamos los determinantes del rendimiento en el examen final de un curso universitario.

### Pregunta 2B.1 — Cargar la base `attend` *(4 pts.)*

In [22]:
# Pregunta 2B.1: Cargar base attend
data("attend")
attend <- na.omit(attend)

### Pregunta 2B.2 — Selección de variables *(4 pts.)*

Subseleccione las siguientes variables:

| Variable | Descripción |
|---|---|
| `attend` | Clases asistidas de un total de 32 |
| `termGPA` | Promedio de notas durante el período |
| `priGPA` | Promedio acumulado antes del período |
| `ACT` | Puntaje en el examen ACT |
| `final` | **Variable dependiente** — puntaje del examen final |
| `hwrte` | Porcentaje de tareas entregadas |
| `frosh` | =1 si es estudiante de primer año |
| `soph` | =1 si es estudiante de segundo año |

In [26]:
base_2B <- subset(attend,
                  select = c(attend, termGPA, priGPA, ACT, final, hwrte, frosh, soph))

### Pregunta 2B.3 — Modelo de regresión múltiple completo *(10 pts.)*

Estime un modelo donde la variable dependiente es `final` y las independientes son todas las demás variables del subconjunto.

In [27]:
# Pregunta 2B.3: Modelo completo con attend_new
modelo_2B <- lm(final ~ attend + termGPA + priGPA + ACT + hwrte + frosh + soph, data = base_2B)
summary(modelo_2B)


Call:
lm(formula = final ~ attend + termGPA + priGPA + ACT + hwrte + 
    frosh + soph, data = base_2B)

Residuals:
     Min       1Q   Median       3Q      Max 
-15.9573  -2.5968  -0.0042   2.6582  11.0815 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 14.27176    1.45667   9.798  < 2e-16 ***
attend      -0.06956    0.04105  -1.694   0.0907 .  
termGPA      3.54151    0.31442  11.264  < 2e-16 ***
priGPA      -0.04149    0.40302  -0.103   0.9180    
ACT          0.27313    0.05031   5.429 7.94e-08 ***
hwrte       -0.01394    0.01043  -1.337   0.1817    
frosh       -0.61310    0.47672  -1.286   0.1989    
soph        -0.82251    0.39459  -2.084   0.0375 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.867 on 666 degrees of freedom
Multiple R-squared:  0.3373,	Adjusted R-squared:  0.3303 
F-statistic: 48.42 on 7 and 666 DF,  p-value: < 2.2e-16


### Pregunta 2B.4 — Bondad de ajuste *(6 pts.)*

Interprete el **R²** y el **R² ajustado** del modelo anterior.

El modelo presenta un R2 de 0.3373, lo que significa que aproximadamente el 33.73% de la variación en el puntaje del examen final (final) es explicada por las variables incluidas en el modelo (attend, termGPA, priGPA, ACT, hwrte, frosh y soph).

Por su parte, el R2 ajustado es 0.3303, lo que indica que, al considerar la cantidad de variables explicativas incorporadas, el modelo sigue explicando cerca del 33.03% de la variación de final.

Como ambos valores son relativamente cercanos, se puede concluir que no hay una pérdida importante de ajuste al corregir por el número de regresores.

In [ ]:
# Pregunta 2B.4: Extraer métricas de bondad de ajuste

### Pregunta 2B.5 — Modelo reducido (excluir variables no significativas) *(10 pts.)*

Genere un nuevo modelo excluyendo las variables con **p-value > 0.05** en el modelo anterior.

In [ ]:
# Identificar variables significativas (p-value <= 0.05)

In [ ]:
# Pregunta 2B.5: Modelo reducido (solo variables significativas)
# Variables significativas identificadas: termGPA, ACT, soph
# (ajuste según los resultados de su modelo)

### Pregunta 2B.6 — Comparación de modelos *(6 pts.)*

Compare el modelo completo y el modelo reducido en términos de **R² ajustado**.

In [ ]:
# Pregunta 2B.6: Comparación final de modelos